## Delta Live Tables

In [0]:
%pip install pybaseball
%pip install numpy
%pip install findspark
%pip install python-dotenv
%pip install --upgrade typing-extensions


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
from pybaseball import statcast
from pybaseball import cache
import datetime
import re
import requests
from dotenv import load_dotenv
import os
import numpy as np
import time
from pyspark.sql.functions import col
from pyspark.sql.types import LongType
from pyspark.sql.types import *
#load_dotenv()

In [0]:
def extract_name(html):
    pattern = re.compile(r'>(.*?)<')
    match = pattern.search(html)
    return match.group(1) if match else None

In [0]:
def load_pitching_stats_info(START_YEAR, END_YEAR):


    if START_YEAR == END_YEAR:
        date_range = pd.date_range(start=f'{START_YEAR}-03-01', end=f'{END_YEAR}-10-31')
        full_df = pd.DataFrame()
        int_columns = ['player', 'wins', 'losses', 'so', 'hbp', 'er', 'sv', 'ibb', 'bb', 'year']
        for date in date_range:

            if datetime.datetime.now().date() > date.date():
                player_id = []
                wins = []
                losses = []
                ip = []
                so = []
                bb = []
                hbp = []
                #    fip = []
                er = []
                #    stuff_plus = []
                sv = []
                war = []
                ibb = []
                name = []
                teams = []
                date = date.strftime("%Y-%m-%d")
                print(date)
                url = f'https://www.fangraphs.com/api/leaders/major-league/data?age=&pos=all&stats=pit&lg=all&qual=0&season={START_YEAR}&season1={END_YEAR}&startdate={date}&enddate={date}&month=1000&pageitems=20000&ind=0&postseason='
                response = requests.get(url)
                print(f'Response {response.status_code}')
                k = response.json()

                if 'data' in k.keys():
                    for row in k['data']:
                        player_id.append(row['xMLBAMID'])
                        wins.append(row['W'])
                        losses.append(row['L'])
                        ip.append(row['IP'])
                        so.append(row['SO'])
                        bb.append(row['BB'])
                        hbp.append(row['HBP'])
                        #       fip.append(row['FIP'])
                        sv.append(row['SV'])
                        er.append(row['ER'])
                        war.append(row['WAR'])
                        ibb.append(row['IBB'])
                        name.append(extract_name(row['Name']))
                        teams.append(extract_name(row['Team']))
                        

                    df_dict = {'player': player_id,
                               'wins': wins,
                               'losses': losses,
                               'ip': ip,
                               'so': so,
                               'bb': bb,
                               'hbp': hbp,
                               'sv': sv,
                               'er': er,
                               'war': war,
                               'ibb': ibb,
                               'name': name,
                               'team': teams}

                    df = pd.DataFrame(df_dict)
                    df['date'] = date
                    df['game_type'] = 'R'
                    full_df = pd.concat([full_df, df])

        playoffs_range = pd.date_range(start=f'{START_YEAR}-10-01', end=f'{END_YEAR}-12-01')
        for date in playoffs_range:

            if datetime.datetime.now().date() > date.date():
                player_id = []
                wins = []
                losses = []
                ip = []
                so = []
                bb = []
                hbp = []
                #    fip = []
                er = []
                #    stuff_plus = []
                sv = []
                war = []
                ibb = []
                name = []
                teams = []

                date = date.strftime("%Y-%m-%d")
                print(date)
                url = f'https://www.fangraphs.com/api/leaders/major-league/data?age=&pos=all&stats=pit&lg=all&qual=0&season={START_YEAR}&season1={END_YEAR}&startdate={date}&enddate={date}&month=1000&pageitems=20000&ind=0&postseason=1'
                response = requests.get(url)
                print(f'Response {response.status_code}')
                k = response.json()

                if 'data' in k.keys():
                    for row in k['data']:
                        player_id.append(row['xMLBAMID'])
                        wins.append(row['W'])
                        losses.append(row['L'])
                        ip.append(row['IP'])
                        so.append(row['SO'])
                        bb.append(row['BB'])
                        hbp.append(row['HBP'])
                        #       fip.append(row['FIP'])
                        sv.append(row['SV'])
                        er.append(row['ER'])
                        war.append(row['WAR'])
                        ibb.append(row['IBB'])
                        name.append(extract_name(row['Name']))
                        teams.append(extract_name(row['Team']))

                    df_dict = {'player': player_id,
                               'wins': wins,
                               'losses': losses,
                               'ip': ip,
                               'so': so,
                               'bb': bb,
                               'hbp': hbp,
                               'sv': sv,
                               'er': er,
                               'war': war,
                               'ibb': ibb,
                               'name': name,
                               'team': teams}

                    df = pd.DataFrame(df_dict)
                    df['date'] = date
                    df['game_type'] = 'P'
                    full_df = pd.concat([full_df, df])


        full_df['year'] = START_YEAR
        for col in int_columns:
            full_df[col] = full_df[col].astype(int)




        return full_df


    else:
        print(f'Only Query 1 full Year worth of data at a time please...')
        raise


In [0]:
def check_for_nan_fields(df, non_null_cols):
    print('Non Null Columns Null Field Count')
    print(df[non_null_cols].isna().sum())

    if df[non_null_cols].isna().sum().sum() > 0:
        print('Non Null Columns Null Field Count')
        print(df[non_null_cols].isna().sum())
        raise Exception('Null Fields Found')
    else:
        print('No Null Fields Found')
        return df
    
def preprocess_data(df):
    df["date"] = pd.to_datetime(df["date"]).dt.date

    return df



In [0]:
pitcher_stats_schema = StructType([

    StructField("player", LongType(), False),
    StructField("wins", IntegerType(), False),
    StructField("losses", IntegerType(), False),
    StructField("ip", DoubleType(), False),
    StructField("so", IntegerType(), False),
    StructField("bb", IntegerType(), False),
    StructField("hbp", IntegerType(), False),
    StructField("sv", IntegerType(), False),
    StructField("er", IntegerType(), False),
    StructField("war", DoubleType(), True),
    StructField("ibb", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("team", StringType(), True),
    StructField("date", DateType(), False),
    StructField("game_type", StringType(), False),
    StructField("year", IntegerType(), False)


])

In [0]:

pitching_stats_2000 = load_pitching_stats_info(2000,2000)

non_nullable_cols = [f.name for f in pitcher_stats_schema.fields if not f.nullable]


check_for_nan_fields(pitching_stats_2000, non_nullable_cols)

pitching_stats_2000 = preprocess_data(pitching_stats_2000)






pitch_stats_spark_df_2000 = spark.createDataFrame(pitching_stats_2000, schema=pitcher_stats_schema)




In [0]:
pitch_stats_spark_df_2000.head(10)

In [0]:
pitch_stats_spark_df_2000 \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year") \
        .option("overwriteSchema", "true") \
        .saveAsTable("mlb_demo.deleta_live.pitching_stats")

In [0]:
%sql
SELECT war, so, ip, wins, name, year, date, team  FROM mlb_demo.deleta_live.pitching_stats
WHERE war = (
  SELECT MAX(war) FROM mlb_demo.deleta_live.pitching_stats
);

In [0]:
%sql

with cte as (

  SELECT  war, so, ip, wins, name, year, date, team , RANK() OVER(PARTITION BY IP ORDER BY WAR DESC) as war_rank
FROM mlb_demo.deleta_live.pitching_stats
where ip > 0


)

SELECT * FROM cte where war_rank = 1
order by ip desc, war desc



## Load 2001 Data

In [0]:

pitching_stats_2001 = load_pitching_stats_info(2001,2001)

non_nullable_cols = [f.name for f in pitcher_stats_schema.fields if not f.nullable]


check_for_nan_fields(pitching_stats_2001, non_nullable_cols)

pitching_stats_2001 = preprocess_data(pitching_stats_2001)






pitch_stats_spark_df_2001 = spark.createDataFrame(pitching_stats_2001, schema=pitcher_stats_schema)




In [0]:
pitch_stats_spark_df_2001 \
        .write \
        .format("delta") \
        .mode("append") \
        .partitionBy("year") \
        .option("mergeSchema", "true") \
        .saveAsTable("mlb_demo.deleta_live.pitching_stats")

In [0]:
%sql

with cte as (

  SELECT  war, so, ip, wins, name, year, date, team , RANK() OVER(PARTITION BY IP ORDER BY WAR DESC) as war_rank
FROM mlb_demo.deleta_live.pitching_stats
where ip > 0
and year = 2001


)

SELECT * FROM cte where war_rank = 1
order by ip desc, war desc



In [0]:
%sq